# Indian agriculture 1966-2017: crop mix and weather dips

**Notebook 4.** This notebook has two parts:

1. **Crop mix.** Farmers didn't just grow more, they also changed *what* they grow. Which crops gained land and which lost it, and where?
2. **Weather dips.** In notebook 2 I saw yearly dips in yield and said I hadn't looked into them. Here I measure them, check them against known drought years, and see which crops are the most unstable.

I use `crop_data_clean.csv` from notebook 1 and the same 8 crops as in notebook 2.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

## Loading the data

In [ ]:
crop_data = pd.read_csv("../data/clean/crop_data_clean.csv")

main_crops = ["Rice", "Wheat", "Maize", "Sorghum", "Pearl millet", "Chickpea", "Groundnut", "Cotton"]

# same steps as notebook 2: 8 crops, only rows where area and production are both known
main_data = crop_data[crop_data["crop"].isin(main_crops)].dropna(subset=["area", "production"])

yearly_totals = main_data.groupby(["crop", "year"])[["area", "production"]].sum().reset_index()
yearly_totals["crop_yield"] = yearly_totals["production"] / yearly_totals["area"] * 1000

yearly_totals.head()

# Part 1: crop mix

## Share of area by crop

I take the area of the 8 crops together as 100% in each year and look at the share of each crop. This only covers these 8 crops (not all land in India), so a share going up means the crop gained *compared to the other 7*.

Order matters in a stacked chart, so I put rice and wheat at the bottom.

In [ ]:
area_by_year = yearly_totals.pivot(index="year", columns="crop", values="area")
area_share = area_by_year.div(area_by_year.sum(axis=1), axis=0) * 100

order = ["Rice", "Wheat", "Maize", "Chickpea", "Cotton", "Groundnut", "Pearl millet", "Sorghum"]
area_share = area_share[order]

area_share.plot.area(figsize=(10, 5), colormap="tab10", alpha=0.9)
plt.title("Share of the area of the 8 crops")
plt.xlabel("year")
plt.ylabel("share (%)")
plt.xlim(1966, 2017)
plt.ylim(0, 100)
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.show()

Same idea as before: I compare the average of the first three years (1966-68) with the last three (2015-17), so one odd year doesn't decide the result.

In [ ]:
share_start = area_share.loc[1966:1968].mean()
share_end = area_share.loc[2015:2017].mean()

share_change = pd.DataFrame({"share 1966-68": share_start, "share 2015-17": share_end})
share_change["change (points)"] = share_change["share 2015-17"] - share_change["share 1966-68"]

share_change.sort_values("change (points)", ascending=False).round(1)

In [ ]:
changes = share_change["change (points)"].sort_values()
colors = ["tomato" if value < 0 else "seagreen" for value in changes]

plt.figure(figsize=(9, 4.5))
plt.barh(changes.index, changes.values, color=colors)
plt.axvline(0, color="black", linewidth=0.8)
plt.title("Change in share of area, 1966-68 to 2015-17")
plt.xlabel("percentage points")
plt.show()

## Where did the change happen?

A national share hides the regions. For each state I work out the share of each crop in that state's area (of the 8 crops), and take the change between 1966-68 and 2015-17. I keep only states with at least 500 (500,000 ha) of these crops in both periods, so small states don't add noise.

In [ ]:
def state_shares(first, last):
    window = main_data[main_data["year"].between(first, last)]

    # average area per year for each state and crop
    area = window.groupby(["state", "crop"])["area"].sum().unstack(fill_value=0) / (last - first + 1)
    return area


start_area = state_shares(1966, 1968)
end_area = state_shares(2015, 2017)

big_states = start_area.index[(start_area.sum(axis=1) >= 500) & (end_area.sum(axis=1) >= 500)]

start_pct = start_area.loc[big_states].div(start_area.loc[big_states].sum(axis=1), axis=0) * 100
end_pct = end_area.loc[big_states].div(end_area.loc[big_states].sum(axis=1), axis=0) * 100

state_change = (end_pct - start_pct)[order]

print(len(big_states), "states")

In [ ]:
plt.figure(figsize=(10, 7))
sns.heatmap(state_change, annot=True, fmt=".0f", cmap="RdYlGn", center=0, cbar_kws={"label": "points"})
plt.title("Change in share of area by state, 1966-68 to 2015-17 (green = gained, red = lost)")
plt.xlabel("")
plt.ylabel("")
plt.show()

## Green Revolution states against the rest

The Green Revolution started in Punjab, Haryana and Uttar Pradesh. I split the data into those three states and all the others, and draw the same stacked chart for each group.

In [ ]:
green_states = ["Punjab", "Haryana", "Uttar Pradesh"]

def share_of_area(data):
    area = data.groupby(["year", "crop"])["area"].sum().unstack()
    return (area.div(area.sum(axis=1), axis=0) * 100)[order]


green_share = share_of_area(main_data[main_data["state"].isin(green_states)])
rest_share = share_of_area(main_data[~main_data["state"].isin(green_states)])

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

green_share.plot.area(ax=axes[0], colormap="tab10", alpha=0.9, legend=False)
axes[0].set_title("Punjab, Haryana, Uttar Pradesh")

rest_share.plot.area(ax=axes[1], colormap="tab10", alpha=0.9)
axes[1].set_title("All other states")
axes[1].legend(bbox_to_anchor=(1.02, 1), loc="upper left")

for ax in axes:
    ax.set_xlabel("year")
    ax.set_xlim(1966, 2017)
    ax.set_ylim(0, 100)
axes[0].set_ylabel("share (%)")

plt.tight_layout()
plt.show()

In [ ]:
compare = pd.DataFrame({
    "Green Rev. 1966-68": green_share.loc[1966:1968].mean(),
    "Green Rev. 2015-17": green_share.loc[2015:2017].mean(),
    "Others 1966-68": rest_share.loc[1966:1968].mean(),
    "Others 2015-17": rest_share.loc[2015:2017].mean(),
})

compare.round(1)

# Part 2: weather dips

## Measuring a dip

A yield that is lower than last year isn't necessarily a bad year, because yields keep rising over time. A real dip is a year that falls **below the trend**. So for each crop and year I take the national yield and compare it with its 5-year average, centred on that year (like in notebook 2):

dip = (yield / 5-year average - 1) * 100

A value of -10 means the yield was 10% below its trend.

The first and last two years don't have a full 5-year window, so for those I use whatever years exist (at least 3). That makes the **first and last years less reliable**, and I keep that in mind when reading the results.

In [ ]:
yield_table = yearly_totals.pivot(index="year", columns="crop", values="crop_yield")[main_crops]

trend = yield_table.rolling(5, center=True, min_periods=3).mean()
dip = (yield_table / trend - 1) * 100

dip.round(1).head(8)

In [ ]:
plt.figure(figsize=(14, 4.5))
sns.heatmap(dip.T, cmap="RdBu", center=0, vmin=-30, vmax=30, cbar_kws={"label": "% from trend"})
plt.title("Yield compared with trend (red = below trend, blue = above)")
plt.xlabel("")
plt.ylabel("")
plt.show()

## Do the bad years match known droughts?

I compared them with published lists of drought years. A Down To Earth article lists 25 major drought years in India since 1873. The ones that fall inside my data (1966-2017) are 1966, 1968, 1972, 1974, 1979, 1982, 1985, 1986, 1987, 2002, 2009, 2014 and 2015. Other sources give shorter lists, for example Scroll has 1966, 1972, 1979, 1987, 2002, 2009, 2014 and 2015, and a government disaster-management document names 1972, 1979, 1987, 2002, 2009, 2014 and 2015.

Sources: [Down To Earth](https://www.downtoearth.org.in/amp/story/climate-change/drought-despite-decades-of-water-conservation-what-a-13-monsoon-deficit-is-exposing), [Scroll](https://scroll.in/article/866262/indias-criteria-for-classifying-drought-hit-regions-are-causing-many-cases-to-go-unreported).

One thing to know: the IMD (India Meteorological Department) now calls such years "deficient" or "large deficient" rather than "drought years", so this is the list of years usually called droughts, not an official IMD label.

To see how widespread a bad year was, I count how many of the 8 crops fell more than 5% below trend, and I also take the average dip across crops.

In [ ]:
bad_years = pd.DataFrame({
    "crops down more than 5%": (dip < -5).sum(axis=1),
    "average dip (%)": dip.mean(axis=1),
})

bad_years.sort_values("average dip (%)").head(10).round(1)

In [ ]:
known_droughts = [1966, 1968, 1972, 1974, 1979, 1982, 1985, 1986, 1987, 2002, 2009, 2014, 2015]

plt.figure(figsize=(12, 4.5))
plt.bar(bad_years.index, bad_years["average dip (%)"], color="grey")

for year in known_droughts:
    plt.bar(year, bad_years.loc[year, "average dip (%)"], color="tomato")

plt.axhline(0, color="black", linewidth=0.8)
plt.title("Average dip across the 8 crops (red = years usually listed as droughts)")
plt.xlabel("year")
plt.ylabel("% below or above trend")
plt.show()

In [ ]:
# how well do the two things match?
worst_ten = bad_years.sort_values("average dip (%)").head(10).index

print("worst 10 years that are on the drought list:", [y for y in worst_ten if y in known_droughts])
print("worst 10 years that are NOT on the list:", [y for y in worst_ten if y not in known_droughts])

print("average dip in drought-list years:", round(bad_years.loc[known_droughts, "average dip (%)"].mean(), 1))
print("average dip in all other years:", round(bad_years.drop(known_droughts)["average dip (%)"].mean(), 1))

In [ ]:
dip.loc[known_droughts].round(0)

## Which crops are the most unstable?

For each crop I take the standard deviation of its dip values. A bigger number means a bigger swing around the trend. I also split the years in two (1966-90 and 1991-2017) to see if crops became more or less stable.

In [ ]:
volatility = pd.DataFrame({
    "all years": dip.std(),
    "1966-90": dip.loc[:1990].std(),
    "1991-2017": dip.loc[1991:].std(),
})
volatility["change"] = volatility["1991-2017"] - volatility["1966-90"]

volatility.sort_values("all years", ascending=False).round(1)

In [ ]:
volatility[["1966-90", "1991-2017"]].sort_values("1991-2017").plot(kind="bar", figsize=(10, 5))
plt.title("How much yield swings around its trend (standard deviation of the dip, %)")
plt.xlabel("")
plt.ylabel("std of dip (%)")
plt.xticks(rotation=0)
plt.show()

These are national numbers, so they are calmer than a single district would be: a drought in one state is partly cancelled by a good year in another. Single districts would be much more unstable.

## What I found

**Crop mix**
- **Wheat gained the most land and sorghum lost the most.** Wheat's share of the area of the 8 crops went from about 13% to 25%. Sorghum went from about 17% to 5%. Pearl millet (-5.7 points) and groundnut (-2.9 points) also lost share, while rice, cotton and maize each gained about 2.5 points.
- **Sorghum was not replaced by the same crop everywhere.** Telangana, Karnataka, Madhya Pradesh, Maharashtra and Gujarat all lost between 15 and 45 points of sorghum share. In Telangana, Maharashtra and Gujarat cotton took its place, in Karnataka it was maize, and in Madhya Pradesh it was wheat.
- **Punjab, Haryana and Uttar Pradesh went heavily into wheat and rice.** Wheat grew from about 33% to 53% of their area and rice from 21% to 34%. Chickpea there fell from 17% to under 2%, and maize and sorghum also fell.
- **The other states moved differently.** There wheat only went from 8% to 16%, while chickpea more than doubled (4.5% to 9.8%) and cotton and maize also gained. So the crop mix moved in opposite directions in the two groups of states.

**Weather dips**
- **The worst years for the 8 crops together are 2015, 2002 and 2009**, with average dips of about 15%, 14% and 12% below trend. Nine of the ten worst years are on published lists of drought years, and the only one that isn't is 1991. The drought-list years had an average dip of about -7.8%, while all other years averaged about +2.6%. I don't know why 1991 is a bad year, because it isn't on any of the lists I found.
- **Rainfed crops swing the most.** Pearl millet (17.6%) and groundnut (14.4%) have the biggest swings around their trend, while rice and wheat swing only about 5%. This fits with rice and wheat being more irrigated, but I didn't test that, because the data has no irrigation information.
- **Not every crop became steadier.** Rice's swings fell after 1990 (6.4% to 4.0%), but groundnut (11.1% to 17.1%), sorghum and cotton became more unstable.

**Limits:** shares are only among these 8 crops, not all cropped land. The first and last years of the dip calculation use a shorter window, so 1966 is less reliable. I matched bad years to published drought lists, and I did not use rainfall data myself, so a dip could also come from floods, pests or other causes.